# Phase 2 — HMM Regime Classifier Validation

Load the trained 5-state HMM model and visualize regime classification
quality: state distributions, transition dynamics, feature separation,
conditional Sharpe ratios, and temporal regime structure.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.hmm_regime import (
    HMMRegimeClassifier, HMMRegimeConfig, RegimeState,
    build_feature_matrix, validate_model,
)

# Load model
MODEL_DIR = '../models/hmm/v1'
clf = HMMRegimeClassifier.load(MODEL_DIR)

# Build test feature matrix (last 2 months of 2024)
PARQUET_DIR = '../data/parquet'
df = pl.read_parquet(os.path.join(PARQUET_DIR, 'year=2024', 'data.parquet'))
ts_max = df['timestamp'].max()
ts_min = df['timestamp'].min()
split = ts_max - int((ts_max - ts_min) * 2/12)
df_test = df.filter(pl.col('timestamp') > split)

config = clf.config
features, timestamps = build_feature_matrix(df_test, config)
states = clf.predict(features)
report = validate_model(clf, features)

print(f'Test samples: {report.n_samples:,}')
print(f'Persistence (5-bar): {report.persistence_5bar:.1%}')
print(f'Log-likelihood: {report.log_likelihood:.2f}')

In [ ]:
# ── State Distribution Pie Chart ──────────────────────────────────
labels = list(report.state_distribution.keys())
sizes = list(report.state_distribution.values())
colors = ['#e74c3c', '#3498db', '#95a5a6', '#f39c12', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%',
    colors=colors, startangle=90
)
ax.set_title('Regime State Distribution (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Transition Matrix Heatmap ─────────────────────────────────────
state_names = [s.name for s in RegimeState]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    report.transition_matrix,
    annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=state_names, yticklabels=state_names,
    ax=ax
)
ax.set_title('Regime Transition Matrix')
ax.set_xlabel('To State')
ax.set_ylabel('From State')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Boxplots by Regime (2x3 grid) ────────────────────────
feature_names = config.feature_columns
state_labels = [s.name for s in states]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for idx, (ax, feat_name) in enumerate(zip(axes.flat, feature_names)):
    data_by_state = {}
    for regime in RegimeState:
        mask = [s == regime for s in states]
        data_by_state[regime.name] = features[mask, idx]

    ax.boxplot(
        [data_by_state[s.name] for s in RegimeState],
        labels=[s.name[:8] for s in RegimeState],
        patch_artist=True,
        boxprops=dict(alpha=0.7),
    )
    ax.set_title(feat_name)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Feature Distributions by Regime', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── State-Conditional Sharpe vs Unconditional ─────────────────────
# Use return_20bar (feature index 5) as a proxy for forward returns
returns = features[:, 5]  # z-scored return_20bar

# Unconditional
uncond_sharpe = np.mean(returns) / max(np.std(returns), 1e-10) * np.sqrt(252 * 390)

# Per-state Sharpe
state_sharpes = {}
for regime in RegimeState:
    mask = np.array([s == regime for s in states])
    r = returns[mask]
    if len(r) > 10:
        state_sharpes[regime.name] = np.mean(r) / max(np.std(r), 1e-10) * np.sqrt(252 * 390)
    else:
        state_sharpes[regime.name] = 0.0

fig, ax = plt.subplots(figsize=(10, 5))
names = list(state_sharpes.keys())
sharpes = list(state_sharpes.values())
bar_colors = ['#e74c3c', '#3498db', '#95a5a6', '#f39c12', '#2ecc71']

bars = ax.bar(names, sharpes, color=bar_colors, alpha=0.8)
ax.axhline(uncond_sharpe, color='black', linestyle='--', label=f'Unconditional ({uncond_sharpe:.2f})')
ax.set_title('Annualized Sharpe Ratio by Regime')
ax.set_ylabel('Sharpe Ratio')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Regime Timeline (color-coded Gantt) ───────────────────────────
# Show a 2-month window of regime classifications
color_map = {
    RegimeState.HIGH_VOL_UP: '#e74c3c',
    RegimeState.HIGH_VOL_DOWN: '#3498db',
    RegimeState.LOW_VOL_RANGE: '#95a5a6',
    RegimeState.BREAKOUT: '#f39c12',
    RegimeState.MEAN_REVERSION: '#2ecc71',
}

# Subsample for readability (every 60th bar ≈ 1 minute)
step = 60
n_show = min(len(states), 50000)
state_arr = np.array([s.value for s in states[:n_show:step]])
x = np.arange(len(state_arr))

fig, ax = plt.subplots(figsize=(16, 3))
for regime in RegimeState:
    mask = state_arr == regime.value
    ax.fill_between(
        x, 0, 1, where=mask,
        color=color_map[regime], alpha=0.8,
        label=regime.name, step='mid'
    )

ax.set_xlim(0, len(state_arr))
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel('Time (minutes, subsampled)')
ax.set_title('Regime Timeline')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5)
plt.tight_layout()
plt.show()